Studiamo come portare un'applicazione dal PC a un server cloud, con un esempio pratico di deployment tramite Photogallery, una piccola applicazione Python scritta con Flask (micro-framework web open-source scritto in Python). In questo md vediamo l'introduzione ad AWS e poi l'installazione a mano su un'istanza EC2.  
Si vede anche la creazione di un'immagine da cui poi creare altre istanze con applicazione già installata e configurata per partire dal boot.

## Infrastruttura AWS
AWS è un provider di cloud pubblico. Offre moltissimi servizi: di calcolo come **EC2** o **Lambda**, di storage come **S3**, di rete come **Route 53** o **CloudFront**, di gestione dei permessi come **IAM** etc..

Per l'infrastruttura tre elementi:
- **Region**: è un'area geografica dove AWS dispone di infrastruttura. Milano e Francoforte sono esempi di regioni europee distinte. Serve per la ridondanza e protezione da guasti su scala geografica più ampia.
- **Availability Zone (AZ)**: è un data center o un insieme di data center all'interno di una regione. Il senso delle AZ è che si possono distribuire più repliche in AZ diverse affinché un guasto locale non comprometta l'intera applicazione.
- **Edge Location**: un punto della rete AWS vicino alla rete. Serve per servizi dove la latenza è importante (es. Content Delivery Network tipo CloudFront).

## Accesso a AWS, strumenti e permessi
Nel **Learner Lab** lavoriamo con una sessione temporanea, con budget di 50 euro a persona e un insieme limitato di servizi. Attento che se spendi tutto il budget allora l'account viene sospeso e **si perde tutto il lavoro fatto**.

In generale, per interagire con AWS diversi strumenti:
- **Web Console** --> si creano e gestiscono le risorse dal browser
- **AWS CLI** --> si gestiscono le risorse con comandi da terminale. Ciò permette di inserire i comandi in uno script
- **CloudShell** --> è un terminale della console web con AWS CLI già disponibile e autenticata
- **SDK** (tipo Boto3 per Python) --> il codice di un'applicazione può invocare direttamente le API AWS (es. leggere un oggetto da S3)
- **Infrastracture as Code** (es. Terraform o OpenTofu) -> si descrive l'infrastruttura in file e lo strumento applica le modifiche necessarie. Si introdurrà alla prossima lez. **Ansible** per automatizzare la configurazione delle macchine

Le istanze EC2 sono **risorse regionali**: se creo un'istanza a Francoforte se poi ci voglio lavorare devo assicurarmi di scegliere la regione corretta.

**IAM** (*Identity and Access Management*) è il servizio che regola i permessi sulle risorse AWS. Si distinguono tre casi:
- **Utente root dell'account AWS** --> ha i privilegi più ampi; si usa per le operazioni a lvl di account
- **Persone che lavorano sull'account** 
- **Applicazioni che chiamano servizi AWS** 

Per le ultime due categorie si possono assegnare i permessi mediante un ruolo IAM. 

## EC2
**Amazon EC2** (*Elastic Compute Cloud*) permette di acquisire capacità di calcolo su richiesta. Si parla di "elastico" perché possiamo acquisire/rilasciare risorse in base alle esigenze. Chiamando istanza ogni server virtuale avviato, la decisione di aggiungere istanze può essere manuale (come vediamo in questa lezione) o automatizzato tramite **Auto Scaling**.

Quando si clicca su **Launch Instance** si devono scegliere più parametri:
- **AMI** (*Amazon Machine Image*): determina con quale sistema operativo e software parte l'istanza. A lezione il prof inizia con una Amazon Linux, ma poi usa un'immagine personalizzata con l'applicazione già installata.
- **Instance type**: quante risorse ha la macchina e quale architettura usa, a lezione sceglie `t3.micro` (1 vCPU, 1 GiB RAM)
- **Volume EBS**: è il disco virtuale dove risiedono SO e file
- **IP pubblico**: l'indirizzo con cui raggiungere la VM
- **Security Group**: regole di firewall che consentono traffico in ingresso/uscita (a lez. porta TCP 22 per SSH e TCP 80 per HTTP pubblico)
- **Key Pair**: come autenticarsi via SSH con una coppia di chiavi

Per collegarsi via SSH:
```bash
chmod 400 chiave.pem
ssh -i chiave.pem ec2-user@IP_PUBBLICO
```
con `chmod 400` chiave privata leggibile solo dal proprietario, `-i` indica quale chiave privata usare.

Quattro modalità per pagare l'istanza:
1. **On-Demand**: piano base e anche il più costoso, si paga l'uso senza impegni a lungo termine.
2. **Spot**: si usa capacità inutilizzata a un prezzo scontato, accettando che AWS possa interrompere l'istanza con pochissimo anticipo (es. se quella capacità è richiesta da un'altra istanza che paga di più) (sensato per fault-tolerant, flexible jobs)
3. **Saving Plans**: si ottiene uno sconto se mi impegno per 1/3 anni a un certo livello di spesa oraria (sensato per steady usage)
4. **Reserved Istances**: si ottiene uno sconto con un impegno da 1 a 3 anni legato a determinate caratteristiche dell'istanza (sensato per steady usage con fixed configuration)

Un esempio in cui ha senso usare Spot è l'addestramento di un modello: si può salvare periodicamente lo stato del training su uno storage persistente --> se l'istanza viene interrotta, una nuova macchina può partire dall'ultimo checkpoint.

### Deployment di Photogallery su una singola istanza
Photogallery v1 mostra un insieme fisso di immagini, c'è il pulsante di upload ma la funzionalità è ancora da implementare. Per ora basta una macchina che esegua Flask e serva la pagina via HTTP.

Per pubblicare l'app su EC2 il prof copia prima la directory dal PC al server via SSH con `scp`, poi entra SSH e installa le dipendenze (Python, Flask, etc.) e avvia l'app:
```bash
scp -i chiave.pem -r photogallery ec2-user@IP_PUBBLICO:/home/ec2-user/
ssh -i chiave.pem ec2-user@IP_PUBBLICO
```
e sull'istanza EC2, dopo essere entrato via SSH:
```bash
sudo dnf install python3-pip 
cd photogallery
sudo pip3 install -r requirements.txt
./run.sh
```
Lo script `run.sh` lancia Flask su `0.0.0.0:80`. Per il browser si usa `http://IP_PUBBLICO/` per vedere l'applicazione.

Inizialmente il browser non mostra la pagina perché il **security group di default è impostato per permettere solo traffico SSH**. Bisogna modificare il security group per consentire traffico HTTP (porta 80) in ingresso.

### Replicare l'applicazione
Con una sola istanza tutto il traffico passa per lo stesso server. Se il carico supera la capacità --> le risposte rallentano. Se l'istanza cade --> indisponibilità. Quindi è necessario scaling orizzontale: eseguire più copie dell'applicazione su istanze diverse.

Ma **se ogni replica ha il proprio indirizzo IP, quale deve contattare il browser?** Due possibilità:
- **Load Balancer**: il client contatta un endpoint comune. Questo riceve la richiesta, seleziona una replica e fa arrivare la risposta al client. Vedremo in questo senso il servizio gestito **Elastic Load Balancing (ELB)** e in particolare un **Application Load Balancer (ALB)**.
- **Bilanciamento via DNS**: la risoluzione dello stesso nome può indirizzare i client verso IP diversi. In questo caso però il controllo sulla distribuzione del traffico è molto limitato.

Photogallery è predisposta per gli **health check**: il balancer può interrogare periodicamente `GET /health` su ciascuna replica.  
Se riceve `200 OK` --> esito positivo.

Il footer della pagina mostra il numero di istanza che ha servito la richiesta. In questo modo si può verificare che le richieste vengano distribuite tra le varie repliche.

Vediamo ora come preparare nuove repliche senza reinstallare tutto a mano.  
Ogni nuova replica deve soddisfare due condizioni: **l'applicazione deve essere installata** e **deve partire automaticamente al boot**.

Il primo modo per l'avvio automatico è tramite il `systemd` di Linux: 
```bash
sudo cp photogallery.service /etc/systemd/system/
sudo systemctl daemon-reload
sudo systemctl enable --now photogallery.service
```
`daemon-reload` fa rileggere a `systemd` le definizioni dei servizi, ed `enable --now` abilita il servizio e lo avvia subito.

Vediamo ora però due modi standard per predisporre l'installazione in macchine nuove: **Custom AMI** e **User Data**.

#### Custom AMI
Si parte da una macchina dove Photogallery è già installata e il servizio è abilitato. Da EC2 si seleziona **Actions > Image and templates > Create image** --> per questa istanza, AWS crea un'AMI associata a uno **snapshot del disco EBS**.

Quando si lancia un'altra istanza da quell'AMI, il suo disco parte con i file, le dipendenze e la configurazione del servizio già presenti.  
Al boot, `systemd` esegue il comando previsto (quindi serve fare il passaggio di `enable` prima di creare l'AMI).  
**L'AMI conserva lo stato del disco, ed è `systemd` che avvia un nuovo processo Flask**.

#### User Data
Si parte invece dall'AMI standard di Amazon Linux e si inserisce uno script nel campo **User Data** al momento del lancio. Nell'esempio lo script è eseguito come root al primo boot di ogni nuova istanza.  
Installa `pip`, scarica ed estrae l'app, installa le dipendenze, copia il file di servizio e lo abilita con `systemctl enable --now`.

Qui è importante la distinzione tra **primo boot** e **boot successivi**: lo script di User Data viene eseguito solo al primo avvio della macchina, a quelli successivi è `systemd` che fa ripartire Photogallery.

<img src="imgs/1.png" width="500" alt="User Data">

In generale i due metodi si possono combinare: ad esempio l'AMI può contenere le dipendenze più pesanti, mentre User Data applica la configurazione specifica di ciascuna istanza.

### Costi e pulizia per il Learner Lab
Il costo della **VM accesa** è solo una voce: possono incidere anche i **volumi EBS** di storage, gli **snapshot**, gli **IPv4 pubblici**, il **traffico in uscita** e i **load balancer**.

Da distinguere tra **Stop** che spegne l'istanza e permette di riaccenderla (quindi il disco EBS rimane e si paga) e **Terminate** che elimina l'istanza e potenzialmente il disco associato in base all'impostazione *Delete on Termination*.

**End Lab** chiude la sessione fermando le istanze EC2, mentre altre risorse come database o NAT gateway possono restare attive.  
Con **Reset del lab** invece si eliminano tutte le risorse del laboratorio senza però ovviamente ripristinare i crediti già consumati.

Per la pulizia si consiglia di **terminare le istanze**, **rimuovere la registrazione delle AMI personalizzate** e **eliminare gli snapshot associati**.